# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

**Dataset Citation:** Mugotitsa, B, Maina, D, Owoko, H, Akinyi, L, Greenfield, J, Todd, J, Kavu, T and Kiragga, A 2026 A FAIR2 Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Frontiers


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# Access key metadata attributes
print(f"Name: {dataset.metadata.name}")
print(f"Description: {dataset.metadata.description}")
print(f"License: {dataset.metadata.license}")
print(f"Published: {dataset.metadata.datePublished}")


## 2. Data Overview
Review available record sets, fields, and their IDs.

We'll list the record sets, their `@id`s, and for each record set, list its fields and columns (with their `@id`).

In [ ]:
# List available record sets and their details
record_sets = dataset.metadata.recordSets
print(f"Record Sets Found: {len(record_sets)}")

for rs in record_sets:
    print(f"Record Set Name: {rs.name} | @id: {rs.id}")
    print("  Fields:")
    for field in rs.fields:
        print(f"    - Field Name: {field.name} | @id: {field.id} | DataType: {field.dataType}")
        if hasattr(field, 'column'):
            print(f"      Column @id: {field.column.id}")
    print()


## 3. Data Extraction
Load data from all available record sets into Pandas DataFrames for analysis.

**Note:** All data extraction references entities (record sets/fields/columns) by their `@id`.

In [ ]:
# Extract data from each record set using their @id
record_set_ids = [rs.id for rs in record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"DataFrame for Record Set @id {record_set_id}: Columns -> {df.columns.tolist()}")
    print(df.head(2))


## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps: filtering records, normalizing numeric fields, grouping data by key attributes.

In this section, we:
- Select a numeric field (e.g., GAD-7 score, PHQ-9 score) by its field `@id`.
- Filter to records with elevated scores.
- Normalize numeric scores.
- Group by a demographic attribute (e.g., gender, village).

All fields/columns are referred to using their `@id`.

In [ ]:
# Choose a record set (the main survey records set):
main_record_set = record_set_ids[0]  # use the first as an example; update if needed
df = dataframes[main_record_set]

# Find numeric fields - look for GAD-7 or PHQ-9 score via field @id
numeric_fields = []
group_fields = []

# Gather candidate fields for numeric and grouping
rs = next(rs for rs in record_sets if rs.id == main_record_set)
for field in rs.fields:
    dt = field.dataType.lower() if isinstance(field.dataType, str) else str(field.dataType).lower()
    if dt in ('float', 'integer', 'number'):
        numeric_fields.append(field.id)
    # Example grouping field: gender, village, level_of_education
    if any(x in field.name.lower() for x in ['gender', 'village', 'education']):
        group_fields.append(field.id)

print(f"Numeric fields found: {numeric_fields}")
print(f"Candidate group fields: {group_fields}")

# Use the first numeric field for demonstration
if numeric_fields:
    numeric_field_id = numeric_fields[0]
    # Threshold for 'elevated' score
    threshold = 10
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    print(filtered_df.head())

    # Normalize the selected numeric field
    normalized_col = f"{numeric_field_id}_normalized"
    filtered_df[normalized_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, normalized_col]].head())

    # Group by the first available group field
    if group_fields:
        group_field_id = group_fields[0]
        if group_field_id in filtered_df.columns:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            print(f"Grouped data by {group_field_id} (mean {numeric_field_id}):")
            print(grouped_df.head())


## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

- Histogram of normalized scores
- Barplot grouped by demographic attribute

**Note:** All field references by `@id`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot normalized distribution of numeric field
if numeric_fields:
    numeric_field_id = numeric_fields[0]
    normalized_col = f"{numeric_field_id}_normalized"
    if normalized_col in filtered_df.columns:
        plt.figure(figsize=(8, 4))
        sns.histplot(filtered_df[normalized_col], bins=20, kde=True)
        plt.title(f"Distribution of Normalized {numeric_field_id}")
        plt.xlabel(normalized_col)
        plt.ylabel("Count")
        plt.show()

# Bar plot of average numeric field by group
if group_fields:
    group_field_id = group_fields[0]
    if group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        plt.figure(figsize=(8, 4))
        sns.barplot(x=group_field_id, y=numeric_field_id, data=grouped_df)
        plt.title(f"Mean {numeric_field_id} by {group_field_id}")
        plt.ylabel(f"Mean {numeric_field_id}")
        plt.xlabel(group_field_id)
        plt.xticks(rotation=45)
        plt.show()


## 6. Conclusion
The FAIR² Mental Health Survey Dataset from Kilifi County provides rich survey data with demographic and assessment results.

- We loaded the dataset using the Croissant schema and explored its metadata.
- All entities were referenced by `@id` (record sets, fields, columns).
- We performed filtering, normalization, and grouping using numeric assessment fields and demographic groupings.
- Visualizations illustrated distributions and relationships across key survey attributes.

This approach serves as a reproducible template for processing Croissant AI-ready datasets using `mlcroissant`.